# 08 Export Gold Chunks for Local RAG

This notebook exports deterministic `gold_chunks.parquet` for the local LexAI runtime.

In [0]:
from pyspark.sql import functions as F, Window
import hashlib
import json
from datetime import datetime, timezone

SOURCE_TABLE = "workspace.default.gold_legal_chunks"
TARGET_DIR = "/Volumes/workspace/legal_data/gold/exports/gold_chunks/latest"
TARGET_FILE = f"{TARGET_DIR}/gold_chunks.parquet"
META_FILE = f"{TARGET_DIR}/export_metadata.json"

REQUIRED_COLS = ["chunk_id", "act_name", "section_number", "chunk_text", "char_count"]

print("[08] SOURCE_TABLE:", SOURCE_TABLE)
print("[08] TARGET_FILE:", TARGET_FILE)

df = spark.table(SOURCE_TABLE).select(*REQUIRED_COLS)

df = (
    df.withColumn("chunk_id", F.trim(F.col("chunk_id").cast("string")))
      .withColumn("act_name", F.trim(F.col("act_name").cast("string")))
      .withColumn("section_number", F.trim(F.col("section_number").cast("string")))
      .withColumn("chunk_text", F.trim(F.col("chunk_text").cast("string")))
      .withColumn("char_count", F.col("char_count").cast("int"))
      .filter(F.col("chunk_id").isNotNull() & (F.col("chunk_id") != ""))
      .filter(F.col("chunk_text").isNotNull() & (F.col("chunk_text") != ""))
)

w = Window.partitionBy("chunk_id").orderBy(
    F.col("act_name").asc_nulls_last(),
    F.col("section_number").asc_nulls_last(),
    F.col("chunk_text").asc_nulls_last(),
)

df = df.withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1).drop("rn")
df = df.orderBy(F.col("chunk_id").asc())

pdf = df.toPandas()

dbutils.fs.mkdirs(TARGET_DIR)
pdf.to_parquet(TARGET_FILE, index=False)

schema_text = "|".join([f"{c}:{str(t)}" for c, t in zip(pdf.columns, pdf.dtypes)])
schema_hash = hashlib.sha256(schema_text.encode("utf-8")).hexdigest()

meta = {
    "source_table": SOURCE_TABLE,
    "target_file": TARGET_FILE,
    "row_count": int(len(pdf)),
    "exported_at_utc": datetime.now(timezone.utc).isoformat(),
    "schema_hash": schema_hash,
    "required_columns": REQUIRED_COLS,
}

dbutils.fs.put(META_FILE, json.dumps(meta, indent=2), overwrite=True)

print("[08] Export completed")
print(json.dumps(meta, indent=2))

[08] SOURCE_TABLE: workspace.default.gold_legal_chunks
[08] TARGET_FILE: /Volumes/workspace/legal_data/gold/exports/gold_chunks/latest/gold_chunks.parquet
Wrote 442 bytes.
[08] Export completed
{
  "source_table": "workspace.default.gold_legal_chunks",
  "target_file": "/Volumes/workspace/legal_data/gold/exports/gold_chunks/latest/gold_chunks.parquet",
  "row_count": 5194,
  "exported_at_utc": "2026-03-03T16:43:27.090556+00:00",
  "schema_hash": "66f2b1d22bd5b21f64bdbd72e69470cfaf94c03a61d4d24613223d4e93afbd31",
  "required_columns": [
    "chunk_id",
    "act_name",
    "section_number",
    "chunk_text",
    "char_count"
  ]
}
